In [5]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    accuracy_score,
    confusion_matrix,
)

In [6]:
np.random.seed(42)


TRAIN_CSV_PATH = "cs-training.csv"
TEST_CSV_PATH = "cs-test.csv"
TARGET_COLUMN = "SeriousDlqin2yrs"

In [7]:
def clean_and_engineer_features(df, income_median, income_col="MonthlyIncome"):
    df = df.copy()

    if df.columns[0].startswith("Unnamed"):
        df = df.drop(columns=[df.columns[0]])

    df[income_col] = df[income_col].fillna(income_median)
    df["NumberOfDependents"] = df["NumberOfDependents"].fillna(0)


    df["total_past_due_incidents"] = (
        df["NumberOfTime30-59DaysPastDueNotWorse"]
        + df["NumberOfTime60-89DaysPastDueNotWorse"]
        + df["NumberOfTimes90DaysLate"]
    )
    df["estimated_monthly_debt"] = df["DebtRatio"] * df[income_col]
    df["income_per_dependent"] = df[income_col] / (df["NumberOfDependents"] + 1)
    df["credit_lines_per_age"] = df["NumberOfOpenCreditLinesAndLoans"] / (
        df["age"] + 1
    )

    return df


feature_columns = [
    "RevolvingUtilizationOfUnsecuredLines",
    "age",
    "NumberOfTime30-59DaysPastDueNotWorse",
    "DebtRatio",
    "MonthlyIncome",
    "NumberOfOpenCreditLinesAndLoans",
    "NumberOfTimes90DaysLate",
    "NumberRealEstateLoansOrLines",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfDependents",
    "total_past_due_incidents",
    "estimated_monthly_debt",
    "income_per_dependent",
    "credit_lines_per_age",
]

In [8]:
print("=" * 60)
print("STEP 1: LOADING TRAINING DATA")
print("=" * 60)

train_df = pd.read_csv(TRAIN_CSV_PATH)
print("Training data shape:", train_df.shape)
print(train_df.head())


income_median = train_df["MonthlyIncome"].median()

train_df = clean_and_engineer_features(train_df, income_median)


clip_caps = {}
for col in ["RevolvingUtilizationOfUnsecuredLines", "DebtRatio"]:
    clip_caps[col] = train_df[col].quantile(0.99)
    train_df[col] = train_df[col].clip(upper=clip_caps[col])


train_df["is_creditworthy"] = 1 - train_df[TARGET_COLUMN]

print("\nClass balance (1 = creditworthy, 0 = not creditworthy):")
print(train_df["is_creditworthy"].value_counts(normalize=True).round(3))


STEP 1: LOADING TRAINING DATA
Training data shape: (150000, 12)
   Unnamed: 0  SeriousDlqin2yrs  RevolvingUtilizationOfUnsecuredLines  age  \
0           1                 1                              0.766127   45   
1           2                 0                              0.957151   40   
2           3                 0                              0.658180   38   
3           4                 0                              0.233810   30   
4           5                 0                              0.907239   49   

   NumberOfTime30-59DaysPastDueNotWorse  DebtRatio  MonthlyIncome  \
0                                     2   0.802982         9120.0   
1                                     0   0.121876         2600.0   
2                                     1   0.085113         3042.0   
3                                     0   0.036050         3300.0   
4                                     1   0.024926        63588.0   

   NumberOfOpenCreditLinesAndLoans  NumberOfTimes90D

In [9]:
X = train_df[feature_columns]
y = train_df["is_creditworthy"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


In [10]:
print("\n" + "=" * 60)
print("STEP 2 & 3: TRAINING MODELS (evaluated on validation split)")
print("=" * 60)

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=6, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, max_depth=10, class_weight="balanced", random_state=42
    ),
}

results = []

for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_val_scaled)
        y_prob = model.predict_proba(X_val_scaled)[:, 1]
    else:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        y_prob = model.predict_proba(X_val)[:, 1]

    accuracy = accuracy_score(y_val, y_pred)
    precision = precision_score(y_val, y_pred)
    recall = recall_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred)
    roc_auc = roc_auc_score(y_val, y_prob)

    results.append({
        "Model": name,
        "Accuracy": round(accuracy, 3),
        "Precision": round(precision, 3),
        "Recall": round(recall, 3),
        "F1-Score": round(f1, 3),
        "ROC-AUC": round(roc_auc, 3),
    })

    print(f"\n--- {name} ---")
    print(f"Accuracy : {accuracy:.3f}")
    print(f"Precision: {precision:.3f}")
    print(f"Recall   : {recall:.3f}")
    print(f"F1-Score : {f1:.3f}")
    print(f"ROC-AUC  : {roc_auc:.3f}")
    print("Confusion Matrix (rows=actual, cols=predicted):")
    print(confusion_matrix(y_val, y_pred))


STEP 2 & 3: TRAINING MODELS (evaluated on validation split)

--- Logistic Regression ---
Accuracy : 0.767
Precision: 0.977
Recall   : 0.768
F1-Score : 0.860
ROC-AUC  : 0.835
Confusion Matrix (rows=actual, cols=predicted):
[[ 1498   507]
 [ 6485 21510]]

--- Decision Tree ---
Accuracy : 0.799
Precision: 0.979
Recall   : 0.801
F1-Score : 0.881
ROC-AUC  : 0.852
Confusion Matrix (rows=actual, cols=predicted):
[[ 1531   474]
 [ 5568 22427]]

--- Random Forest ---
Accuracy : 0.837
Precision: 0.977
Recall   : 0.846
F1-Score : 0.907
ROC-AUC  : 0.864
Confusion Matrix (rows=actual, cols=predicted):
[[ 1440   565]
 [ 4316 23679]]


In [11]:
print("\n" + "=" * 60)
print("FINAL MODEL COMPARISON (on validation split)")
print("=" * 60)

results_df = pd.DataFrame(results).sort_values(by="ROC-AUC", ascending=False)
print(results_df.to_string(index=False))

best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]
print(f"\nBest performing model based on ROC-AUC: {best_model_name}")


FINAL MODEL COMPARISON (on validation split)
              Model  Accuracy  Precision  Recall  F1-Score  ROC-AUC
      Random Forest     0.837      0.977   0.846     0.907    0.864
      Decision Tree     0.799      0.979   0.801     0.881    0.852
Logistic Regression     0.767      0.977   0.768     0.860    0.835

Best performing model based on ROC-AUC: Random Forest


In [12]:
print("\n" + "=" * 60)
print("FEATURE IMPORTANCE (from Random Forest)")
print("=" * 60)

rf_model = models["Random Forest"]
importance_df = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": rf_model.feature_importances_,
}).sort_values(by="Importance", ascending=False)
print(importance_df.to_string(index=False))


FEATURE IMPORTANCE (from Random Forest)
                             Feature  Importance
            total_past_due_incidents    0.263851
RevolvingUtilizationOfUnsecuredLines    0.253918
             NumberOfTimes90DaysLate    0.104738
NumberOfTime30-59DaysPastDueNotWorse    0.096085
NumberOfTime60-89DaysPastDueNotWorse    0.054462
                                 age    0.046432
                credit_lines_per_age    0.034787
                           DebtRatio    0.028294
              estimated_monthly_debt    0.026394
                income_per_dependent    0.025910
                       MonthlyIncome    0.023265
     NumberOfOpenCreditLinesAndLoans    0.022065
        NumberRealEstateLoansOrLines    0.014117
                  NumberOfDependents    0.005681


In [13]:
print("\n" + "=" * 60)
print("STEP 7: PREDICTING ON cs-test.csv (unlabeled)")
print("=" * 60)

test_df = pd.read_csv(TEST_CSV_PATH)
print("Test data shape:", test_df.shape)


if test_df.columns[0].startswith("Unnamed"):
    test_ids = test_df[test_df.columns[0]]
else:
    test_ids = pd.Series(range(1, len(test_df) + 1))


test_df_clean = clean_and_engineer_features(test_df, income_median)

for col, cap in clip_caps.items():
    test_df_clean[col] = test_df_clean[col].clip(upper=cap)

X_test_final = test_df_clean[feature_columns]


if best_model_name == "Logistic Regression":
    X_test_final_input = scaler.transform(X_test_final)
else:
    X_test_final_input = X_test_final

test_predictions = best_model.predict(X_test_final_input)
test_probabilities = best_model.predict_proba(X_test_final_input)[:, 1]

submission = pd.DataFrame({
    "Id": test_ids,
    "Predicted_Creditworthy": test_predictions,       # 1 = good, 0 = bad
    "Probability_Creditworthy": test_probabilities.round(4),
})

submission.to_csv("submission.csv", index=False)

print(f"\nPredictions generated using: {best_model_name}")
print(submission.head(10))
print("\nSaved full predictions to 'submission.csv'")


STEP 7: PREDICTING ON cs-test.csv (unlabeled)
Test data shape: (101503, 12)

Predictions generated using: Random Forest
   Id  Predicted_Creditworthy  Probability_Creditworthy
0   1                       1                    0.5613
1   2                       1                    0.6794
2   3                       1                    0.8160
3   4                       1                    0.5190
4   5                       0                    0.4116
5   6                       1                    0.7442
6   7                       1                    0.6052
7   8                       1                    0.6107
8   9                       1                    0.9466
9  10                       0                    0.2997

Saved full predictions to 'submission.csv'
